In [1]:
import pandas as pd
df_bm25 = pd.read_parquet("bm25_top10.parquet")
df_sen_transformer = pd.read_parquet("sen_transformer_top_10.parquet")

In [2]:
df_bm25

,123839,188629,13898,316959,515031,123783,4332,9591,114982,563603,...,66818,862773,11326,896124,12319,4421,296526,341793,270009,110149
0,806300,2185399,13898,1326263,888195,1210005,989116,2416624,2305139,390174,...,66818,1395771,2047737,906780,12319,566613,802604,1447150,empty,empty
1,123839,1990307,2213954,930698,887826,123783,1442076,908435,1341103,2207107,...,358047,1671141,11326,1720286,276196,567146,1930537,1901595,empty,empty
2,806326,188629,2108627,316959,515031,1209576,4332,138626,2305216,373237,...,1588006,1395725,1630363,1938239,1670681,1128206,561052,341793,empty,empty
3,836567,2191096,1921403,348869,1176215,1675282,1815319,390102,2444081,12846,...,889006,1395776,1224804,1500863,1778275,1438630,1584384,1596837,empty,empty
4,806075,1919943,1874948,1836641,1569953,1630942,1991105,1898638,1191721,349302,...,751723,1434303,1337294,1209146,1675166,2278146,128653,1534668,empty,empty
5,1793430,2304431,2216860,2156815,1125994,1214652,1090555,2250838,917379,563603,...,28752,1406415,2202318,380630,81009,1450334,1295704,398261,empty,empty
6,806263,1046396,1698181,881673,836749,2200269,1516841,143444,2303696,394294,...,309952,862773,1338231,1782146,2086809,381372,2195949,2240196,empty,empty
7,799188,2296288,1440180,2215542,832086,579503,116674,1966629,1672157,270469,...,359824,1551102,1137830,561358,1778752,1861483,2054046,585065,empty,empty
8,1727316,2129645,2248373,1525222,1382041,419894,1903340,1483036,2304325,1233547,...,386220,1395758,1542023,2312223,646010,920305,2089089,1542968,empty,empty
9,1901730,2290269,2336616,1465433,268224,810375,928308,1759147,empty,219324,...,552255,1377126,1768494,1900646,2086621,1630486,822322,581557,empty,empty


In [3]:
df_sen_transformer

,123839,188629,13898,316959,515031,123783,4332,9591,114982,563603,...,37970,2612,66818,862773,11326,896124,12319,4421,296526,341793
0,516871,188629,13898,316959,515031,2267777,4332,9591,725799,251631,...,37970,746818,66818,862773,1161272,747395,891422,1786733,1633820,341793
1,647478,251586,785960,562691,706326,421974,615358,143444,1454057,12846,...,580876,1137023,386220,1426706,11326,2054921,12319,1297589,718007,1857381
2,2129172,2397950,1382760,1326263,780987,1630942,1712050,1898638,1478113,273547,...,177545,2612,287026,865851,1517411,573381,276196,1111917,1901019,576283
3,806300,2061524,941146,2156815,832086,1209576,2361455,595391,2304325,563603,...,484539,1084496,2387535,2272303,623784,538460,2073704,365980,420896,2303264
4,2357011,272146,802532,687712,842375,2322923,1297050,1445185,294327,840695,...,736630,1963277,532111,189807,1636981,1038074,1119295,1554833,296526,576336
5,1222510,1948304,1048798,42143,860147,152040,1878340,2378793,2352224,2091601,...,154939,1089914,242676,1964415,1210930,561358,2086621,1224508,128653,2197644
6,806263,2437665,1989378,1639962,828931,116684,1438336,1472431,95752,2307963,...,208832,348197,309952,319262,1207519,2253573,1670681,1584239,1901020,1174734
7,2433706,1278316,1874948,1469362,833668,2117771,2342190,494103,2352254,2146572,...,2313722,1304252,1376367,2126545,272778,896124,373696,2271988,194498,876840
8,708880,1614942,2031249,729083,754830,1411855,774950,2021787,102670,2152054,...,1110574,858691,889006,275965,1901218,2121565,173798,2283929,1438397,576278
9,775424,1363774,462902,1422464,823332,123783,1913962,863960,2298231,1674515,...,718515,353802,775005,1653529,653832,2331828,2110002,1227506,1265304,355461


In [7]:
def formula(index: int) -> float:
    return 1 / (60 + index)

In [8]:
def reciprocal_rank_fusion(bm25: list, sen_transformer: list) -> list:
    set_of_examined = set()
    tuple_list = []

    for i,doc_id in enumerate(bm25):
        if doc_id != "empty":
            set_of_examined.add(doc_id)

            doc_score = 0

            doc_score += formula(i + 1)

            for j in range(len(sen_transformer)):
                if sen_transformer[j] == doc_id:
                    doc_score += formula(j + 1)
                    break

            tuple_list.append((doc_score, doc_id))

    for i in range(len(sen_transformer)):
        doc_id = sen_transformer[i]

        if doc_id not in set_of_examined:
            set_of_examined.add(doc_id)
            tuple_list.append((formula(i + 1), doc_id))

    tuple_list.sort(reverse=True)
    return tuple_list

In [9]:
import ir_datasets
dataset = ir_datasets.load("wikir/en1k/training")

In [10]:
query_ids = [query.query_id for query in dataset.queries_iter()]
query_ids

['123839',
 '188629',
 '13898',
 '316959',
 '515031',
 '123783',
 '4332',
 '9591',
 '114982',
 '563603',
 '104206',
 '1580851',
 '23678',
 '37480',
 '109454',
 '544250',
 '12519',
 '115015',
 '84287',
 '24998',
 '11925',
 '643',
 '113075',
 '1250387',
 '72012',
 '167865',
 '1425120',
 '24883',
 '27200',
 '136856',
 '6031',
 '1313191',
 '187115',
 '84110',
 '208234',
 '6431',
 '10799',
 '1417144',
 '32512',
 '677637',
 '97937',
 '113765',
 '12900',
 '1294909',
 '73745',
 '11692',
 '22173',
 '76662',
 '164301',
 '23344',
 '9367',
 '592217',
 '98546',
 '98175',
 '15189',
 '1256',
 '1815937',
 '1249076',
 '80274',
 '38489',
 '11311',
 '4754',
 '257711',
 '141668',
 '100895',
 '32945',
 '112321',
 '133725',
 '424464',
 '523456',
 '1327515',
 '12101',
 '9739',
 '189869',
 '143444',
 '78940',
 '12527',
 '327199',
 '7707',
 '97455',
 '1262433',
 '488875',
 '74026',
 '358108',
 '95179',
 '7862',
 '151797',
 '92980',
 '100876',
 '238228',
 '107755',
 '32539',
 '12953',
 '24843',
 '119528',
 '307

In [11]:
from collections import defaultdict
query_and_hybrid_scores = defaultdict(list)

for query_id in query_ids:
    query_and_hybrid_scores[query_id] = reciprocal_rank_fusion(
        bm25=list(df_bm25[query_id]),
        sen_transformer=list(df_sen_transformer[query_id])
    )

query_and_hybrid_scores = dict(query_and_hybrid_scores)

In [12]:
query_and_hybrid_scores

{'123839': [(0.032018442622950824, '806300'),
  (0.029850746268656716, '806263'),
  (0.01639344262295082, '516871'),
  (0.016129032258064516, '647478'),
  (0.016129032258064516, '123839'),
  (0.015873015873015872, '806326'),
  (0.015873015873015872, '2129172'),
  (0.015625, '836567'),
  (0.015384615384615385, '806075'),
  (0.015384615384615385, '2357011'),
  (0.015151515151515152, '1793430'),
  (0.015151515151515152, '1222510'),
  (0.014705882352941176, '799188'),
  (0.014705882352941176, '2433706'),
  (0.014492753623188406, '708880'),
  (0.014492753623188406, '1727316'),
  (0.014285714285714285, '775424'),
  (0.014285714285714285, '1901730')],
 '188629': [(0.032266458495966696, '188629'),
  (0.01639344262295082, '2185399'),
  (0.016129032258064516, '251586'),
  (0.016129032258064516, '1990307'),
  (0.015873015873015872, '2397950'),
  (0.015625, '2191096'),
  (0.015625, '2061524'),
  (0.015384615384615385, '272146'),
  (0.015384615384615385, '1919943'),
  (0.015151515151515152, '230443

In [13]:
query_and_hybrid_scores["123839"]

[(0.032018442622950824, '806300'),
 (0.029850746268656716, '806263'),
 (0.01639344262295082, '516871'),
 (0.016129032258064516, '647478'),
 (0.016129032258064516, '123839'),
 (0.015873015873015872, '806326'),
 (0.015873015873015872, '2129172'),
 (0.015625, '836567'),
 (0.015384615384615385, '806075'),
 (0.015384615384615385, '2357011'),
 (0.015151515151515152, '1793430'),
 (0.015151515151515152, '1222510'),
 (0.014705882352941176, '799188'),
 (0.014705882352941176, '2433706'),
 (0.014492753623188406, '708880'),
 (0.014492753623188406, '1727316'),
 (0.014285714285714285, '775424'),
 (0.014285714285714285, '1901730')]

In [15]:
len(query_and_hybrid_scores.keys())

1444

### Cross Encoder

In [14]:
from sentence_transformers import CrossEncoder
model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L6-v2')

/run/media/tahas44/Yeni Birim/Technarts/Intern/NLP/InformationRetrieval/.venv/lib64/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 2754.10it/s]


In [16]:
query_id_vs_text = defaultdict(str)

for query in dataset.queries_iter():
    query_id_vs_text[query.query_id] = query.text

query_id_vs_text = dict(query_id_vs_text)

doc_id_vs_text = defaultdict(str)

for doc in dataset.docs_iter():
    doc_id_vs_text[doc.doc_id] = doc.text

doc_id_vs_text = dict(doc_id_vs_text)

In [17]:
query_doc_pairs = defaultdict(list)

for query_id in query_ids:
    for mytuple in query_and_hybrid_scores[query_id]:
        query_doc_pairs[query_id].append((query_id_vs_text[query_id], doc_id_vs_text[mytuple[1]]))

query_doc_pairs = dict(query_doc_pairs)

In [19]:
query_score_dict = defaultdict(list)

for query_id in query_ids:
    query_score_dict[query_id] = list(model.predict(query_doc_pairs[query_id]))

query_doc_pairs = dict(query_doc_pairs)

In [20]:
query_score_dict["123839"]

[np.float32(6.4627924),
 np.float32(4.384878),
 np.float32(-0.4506746),
 np.float32(-0.17859715),
 np.float32(6.419842),
 np.float32(4.9677753),
 np.float32(-0.33984625),
 np.float32(3.0357656),
 np.float32(5.5189123),
 np.float32(0.458524),
 np.float32(2.3268332),
 np.float32(2.2397873),
 np.float32(0.75131375),
 np.float32(1.5397115),
 np.float32(-2.3310945),
 np.float32(-3.6361036),
 np.float32(-1.3943533),
 np.float32(-4.7235107)]

In [22]:
retrieved_docs_with_scores = defaultdict(list)

for query_idx in range(len(query_ids)):
    tuple_list = []
    query_id = query_ids[query_idx]

    for doc_idx in range(len(query_and_hybrid_scores[query_id])):
        tuple_list.append((query_score_dict[query_id][doc_idx], query_and_hybrid_scores[query_id][doc_idx][1]))

    retrieved_docs_with_scores[query_id] = sorted(tuple_list, reverse=True)

retrieved_docs_with_scores = dict(retrieved_docs_with_scores)

In [24]:
most_related_10_with_scores = defaultdict(list)

for query_id in query_ids:
    most_related_10_with_scores[query_id] = retrieved_docs_with_scores[query_id][:10]

most_related_10_with_scores = dict(most_related_10_with_scores)

In [26]:
most_related_5_with_scores = defaultdict(list)

for query_id in query_ids:
    most_related_5_with_scores[query_id] = retrieved_docs_with_scores[query_id][:5]

most_related_5_with_scores = dict(most_related_5_with_scores)

In [27]:
qrels_dict = defaultdict(list)

for qrel in dataset.qrels_iter():
    qrels_dict[qrel.query_id].append(qrel.doc_id)

qrels_dict = dict(qrels_dict)

query_ids = [query.query_id for query in dataset.queries_iter()]

class Scoredoc:
    def __init__(self, doc_id, score):
        self.doc_id = doc_id
        self.score = score

score_doc_dict = defaultdict(list)

for scoreddoc in dataset.scoreddocs_iter():
    doc_id = scoreddoc.doc_id
    score = scoreddoc.score

    scoreddoc_object = Scoredoc(doc_id, score)

    score_doc_dict[scoreddoc.query_id].append(scoreddoc_object)

In [28]:
from collections import defaultdict
from sklearn.metrics import ndcg_score
import numpy as np
import pandas as pd

def recall(found_list: list, test_list: list) -> float:
    counter = 0

    test_set = set(test_list)

    for mytuple in found_list:
        if mytuple[1] in test_set:
            counter += 1

    return (counter / len(test_list)) * 100

def precision(found_list: list, test_list: list) -> float:
    if len(found_list) == 0:
        return 0

    counter = 0

    test_set = set(test_list)

    for mytuple in found_list:
        if mytuple[1] in test_set:
            counter += 1

    return (counter / len(found_list)) * 100

def precision_AP(found_docs: list, test_list: list) -> float:
    counter = 0

    test_set = set(test_list)

    for mytuple in found_docs:
        if mytuple[1] in test_set:
            counter += 1

    return counter / len(found_docs)

def AP(found_docs: list, test_list: list, value: int) -> float:
    total = 0

    for i in range(1, value + 1):
        if i < len(found_docs):
            precision_k = precision_AP(found_docs[:i], test_list)
            total += precision_k * (found_docs[i - 1][1] in set(test_list))

    return total / len(test_list)

def get_ndcg_list(most_dict: dict, value: int, query_ids: list, score_doc_dict: dict) -> list:
    doc_id_score_dict = defaultdict(float)
    ndcg_list = []

    for query_id in query_ids:
        scoredoc_object_list = score_doc_dict[query_id]

        for scoreddoc_object in scoredoc_object_list:
            doc_id_score_dict[scoreddoc_object.doc_id] = scoreddoc_object.score

        model_score_tuple_list = most_dict[query_id]
        y_score, y_true = [], []

        for mytuple in model_score_tuple_list:
            doc_id = mytuple[1]
            score = mytuple[0]

            y_score.append(score)
            y_true.append(doc_id_score_dict[doc_id])

        if len(y_score) == 0:
            ndcg_list.append(0.0)
            continue

        if len(y_score) == 1:
            y_true.append(0.0)
            y_score.append(0.0)

        if len(y_true) == len(y_score):
            ndcg_list.append(ndcg_score(np.asarray([y_true]), np.asarray([y_score]), k=value))
        else:
            print("There is a problem with query", query_id)

    return ndcg_list

- for top 10

In [29]:
recall_10_list = []

for query_id in query_ids:
    recall_10_list.append(recall(most_related_10_with_scores[query_id], qrels_dict[query_id]))

precision_10_list = []

for query_id in query_ids:
    precision_10_list.append(precision(most_related_10_with_scores[query_id], qrels_dict[query_id]))

AP_10_list = []

for query_id in query_ids:
    AP_10_list.append(AP(most_related_10_with_scores[query_id], qrels_dict[query_id], 10))

NDCG_10_list = get_ndcg_list(most_related_10_with_scores, 10, query_ids, score_doc_dict)

- for top 5

In [30]:
recall_5_list = []

for query_id in query_ids:
    recall_5_list.append(recall(most_related_5_with_scores[query_id], qrels_dict[query_id]))

precision_5_list = []

for query_id in query_ids:
    precision_5_list.append(precision(most_related_5_with_scores[query_id], qrels_dict[query_id]))

AP_5_list = []

for query_id in query_ids:
    AP_5_list.append(AP(most_related_5_with_scores[query_id], qrels_dict[query_id], 5))

NDCG_5_list = get_ndcg_list(most_related_5_with_scores, 5, query_ids, score_doc_dict)

In [31]:
def base_df(
        query_ids: list,
        recall_5: list,
        precision_5: list,
        AP_5: list,
        NDCG_5: list,
        recall_10: list,
        precision_10: list,
        AP_10: list,
        NDCG_10: list
) -> pd.DataFrame:

    df = pd.DataFrame({
        "Query_ID": query_ids,
        "recall_5": recall_5,
        "precision_5": precision_5,
        "AP_5": AP_5,
        "NDCG_5": NDCG_5,
        "recall_10": recall_10,
        "precision_10": precision_10,
        "AP_10": AP_10,
        "NDCG_10": NDCG_10
    })

    df["f_score_5"] = 2 * df["recall_5"] * df["precision_5"] / (df["recall_5"] + df["precision_5"])
    df["f_score_10"] = 2 * df["recall_10"] * df["precision_10"] / (df["recall_10"] + df["precision_10"])

    df["f_score_5"] = df["f_score_5"].fillna(0)
    df["f_score_10"] = df["f_score_10"].fillna(0)

    return df

In [32]:
df = base_df(
    query_ids=query_ids,
    recall_5=recall_5_list,
    precision_5=precision_5_list,
    AP_5=AP_5_list,
    NDCG_5=NDCG_5_list,
    recall_10=recall_10_list,
    precision_10=precision_10_list,
    AP_10=AP_10_list,
    NDCG_10=NDCG_10_list
)

df

,Query_ID,recall_5,precision_5,AP_5,NDCG_5,recall_10,precision_10,AP_10,NDCG_10,f_score_5,f_score_10
0,123839,66.666667,80.0,0.500000,0.998041,100.000000,60.0,0.915079,0.991199,72.727273,75.000000
1,188629,33.333333,40.0,0.277778,0.990272,33.333333,20.0,0.277778,0.987397,36.363636,25.000000
2,13898,50.000000,60.0,0.500000,0.000000,50.000000,30.0,0.500000,0.000000,54.545455,37.500000
3,316959,22.222222,40.0,0.222222,0.989511,22.222222,20.0,0.222222,0.987374,28.571429,21.052632
4,515031,7.142857,20.0,0.071429,0.688473,14.285714,20.0,0.087302,0.714439,10.526316,16.666667
...,...,...,...,...,...,...,...,...,...,...,...
1439,896124,25.000000,40.0,0.145833,0.970551,25.000000,20.0,0.145833,0.976072,30.769231,22.222222
1440,12319,4.545455,20.0,0.045455,0.877215,4.545455,10.0,0.045455,0.823364,7.407407,6.250000
1441,4421,0.000000,0.0,0.000000,0.986912,0.000000,0.0,0.000000,0.971606,0.000000,0.000000
1442,296526,10.000000,20.0,0.033333,0.898832,20.000000,20.0,0.033333,0.893007,13.333333,20.000000


In [33]:
df.isna().sum()

Query_ID        0
recall_5        0
precision_5     0
AP_5            0
NDCG_5          0
recall_10       0
precision_10    0
AP_10           0
NDCG_10         0
f_score_5       0
f_score_10      0
dtype: int64

In [34]:
def create_parquet_df(method_name: str, df: pd.DataFrame) -> pd.DataFrame:
    mydict = {
        "Method": method_name,
        "recall_5_mean": df["recall_5"].mean(),
        "recall_5_std": df["recall_5"].std(),
        "recall_5_max": df["recall_5"].max(),
        "recall_5_min": df["recall_5"].min(),
        "recall_10_mean": df["recall_10"].mean(),
        "recall_10_std": df["recall_10"].std(),
        "recall_10_max": df["recall_10"].max(),
        "recall_10_min": df["recall_10"].min(),
        "precision_5_mean": df["precision_5"].mean(),
        "precision_5_std": df["precision_5"].std(),
        "precision_5_max": df["precision_5"].max(),
        "precision_5_min": df["precision_5"].min(),
        "precision_10_mean": df["precision_10"].mean(),
        "precision_10_std": df["precision_10"].std(),
        "precision_10_max": df["precision_10"].max(),
        "precision_10_min": df["precision_10"].min(),
        "f_score_5_mean": df["f_score_5"].mean(),
        "f_score_5_std": df["f_score_5"].std(),
        "f_score_5_max": df["f_score_5"].max(),
        "f_score_5_min": df["f_score_5"].min(),
        "f_score_10_mean": df["f_score_10"].mean(),
        "f_score_10_std": df["f_score_10"].std(),
        "f_score_10_max": df["f_score_10"].max(),
        "f_score_10_min": df["f_score_10"].min(),
        "MAP_5": df["AP_5"].mean(),
        "MAP_10": df["AP_10"].mean(),
        "NDCG_5_mean": df["NDCG_5"].mean(),
        "NDCG_5_std": df["NDCG_5"].std(),
        "NDCG_5_max": df["NDCG_5"].max(),
        "NDCG_5_min": df["NDCG_5"].min(),
        "NDCG_10_mean": df["NDCG_10"].mean(),
        "NDCG_10_std": df["NDCG_10"].std(),
        "NDCG_10_max": df["NDCG_10"].max(),
        "NDCG_10_min": df["NDCG_10"].min()
    }

    df_parquet = pd.DataFrame(mydict, index=[0])

    return df_parquet

In [35]:
df_parquet = create_parquet_df("Hybrid Search", df)

In [36]:
df_parquet

,Method,recall_5_mean,recall_5_std,recall_5_max,recall_5_min,recall_10_mean,recall_10_std,recall_10_max,recall_10_min,precision_5_mean,...,MAP_5,MAP_10,NDCG_5_mean,NDCG_5_std,NDCG_5_max,NDCG_5_min,NDCG_10_mean,NDCG_10_std,NDCG_10_max,NDCG_10_min
0,Hybrid Search,17.071468,14.247436,83.333333,0.0,23.777552,20.117251,100.0,0.0,35.817175,...,0.131637,0.170402,0.784984,0.33035,1.0,0.0,0.789487,0.288096,1.0,0.0


In [37]:
df_parquet.to_parquet("hybrid_search.parquet")